# Imports

In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import (
    col, lit, when, expr,
    to_timestamp, to_date,
    concat_ws,
    dayofweek, hour,
    avg, stddev_pop, count, countDistinct,
    window, from_json, udf, abs as sql_abs)

# Java Setup

In [2]:
!apt-get install -y openjdk-17-jre 2>/dev/null > /dev/null

# Download Dataset

In [3]:
!wget -q -O taxi_rides_1pc.csv.gz \
https://www.dropbox.com/scl/fi/v8ei5laqcalrx30z3lsty/taxi_rides_1pc.csv.gz?rlkey=q1lq7l56c4j97h9kymsdroau5&dl=0

# Install Kafka

In [4]:
%%bash
KAFKA_VERSION=3.7.2
KAFKA=kafka_2.12-$KAFKA_VERSION

pkill -f kafka.Kafka || true
pkill -f kafka-server-start || true

if [ ! -d "$KAFKA" ]; then
  wget -q -O /tmp/$KAFKA.tgz https://dlcdn.apache.org/kafka/$KAFKA_VERSION/$KAFKA.tgz
  tar xfz /tmp/$KAFKA.tgz
fi

wget -q -O $KAFKA/config/server1.properties \
https://github.com/smduarte/spbd-2526/raw/refs/heads/main/docs/labs/projs/server1.properties

rm -rf /tmp/kraft-combined-logs

UUID=`$KAFKA/bin/kafka-storage.sh random-uuid`
$KAFKA/bin/kafka-storage.sh format -t $UUID -c $KAFKA/config/server1.properties
$KAFKA/bin/kafka-server-start.sh -daemon $KAFKA/config/server1.properties

jps


Formatting /tmp/kraft-combined-logs with metadata.version 3.7-IV4.
2686 Unknown
2687 Jps


## Kafka publisher

In [5]:
!pip --quiet install kafka-python dataclasses
!wget -q -O kafka-publisher.py \
https://raw.githubusercontent.com/smduarte/spbd-2526/refs/heads/main/docs/labs/projs/kafka-publisher.py

!nohup python kafka-publisher.py --topic taxis_json --speedup 60 --filename taxi_rides_1pc.csv.gz 2> /dev/null &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 9.8 MB/s eta 0:00:00


## Kafka Session

This block initializes a new SparkSession with Kafka support, ensuring that Spark can consume data from a real-time stream.


In [6]:
try:
    spark.stop()
except:
    pass

kafka_pkg = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1",
    "org.apache.spark:spark-token-provider-kafka-0-10_2.13:4.0.1"
])

spark = (SparkSession.builder
    .appName("SPBD2526-Project2")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", kafka_pkg)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

Spark: 4.0.1


# Grid UDF

This code defines a spatial grid over the New York area and maps latitude/longitude coordinates into discrete grid cells.

In [7]:
MIN_LON = -74.916578
MAX_LAT = 41.47718278
LON_DELTA = 0.005986
LAT_DELTA = 0.004491556

def latlon_to_grid(lat, lon):
    return (int((MAX_LAT - lat) / LAT_DELTA), int((lon - MIN_LON) / LON_DELTA))

grid_udf = udf(
    lambda lat, lon: latlon_to_grid(lat, lon),
    StructType([StructField("row", IntegerType(), True),
                StructField("col", IntegerType(), True)]))

# **Do taxi rides exhibit geographic and/or temporal patterns?**

In the first assignment, the analysis was conducted exclusively over an archived dataset of New York City taxi rides from 2013. This batch-oriented analysis allowed us to identify stable, long-term patterns in the data, such as the most frequent pickup and dropoff areas, the most common routes, and temporal distributions of rides across hours of the day, days of the week, and months of the year.

While this historical analysis is valuable to understand what is typical or expected, it does not provide any information about what is happening right now. In contrast, the second assignment introduces a real-time data stream, where taxi ride events are continuously generated and made available through Apache Kafka, simulating live taxi activity.

By incorporating streaming data, we can observe short-term dynamics that are not visible in archived data alone. This enables analyses such as detecting emerging hotspots, identifying abnormal activity compared to historical baselines, and monitoring real-time deviations in demand, revenue, or tipping behavior. The objective is not to replace the historical analysis, but to enrich it by combining long-term patterns with real-time observations.

# Stream Schema

This block defines the schema for incoming taxi ride events from the Kafka stream. By explicitly specifying field types, Spark can correctly parse the JSON data and efficiently process it in streaming mode, ensuring consistent handling of spatial and temporal features.

In [8]:
taxi_schema = StructType([
    StructField("medallion", StringType()),
    StructField("hack_license", StringType()),
    StructField("pickup_datetime", StringType()),
    StructField("dropoff_datetime", StringType()),
    StructField("trip_time_in_secs", IntegerType()),
    StructField("trip_distance", DoubleType()),
    StructField("pickup_longitude", DoubleType()),
    StructField("pickup_latitude", DoubleType()),
    StructField("dropoff_longitude", DoubleType()),
    StructField("dropoff_latitude", DoubleType()),
    StructField("payment_type", StringType()),
    StructField("fare_amount", DoubleType()),
    StructField("surcharge", DoubleType()),
    StructField("mta_tax", DoubleType()),
    StructField("tip_amount", DoubleType()),
    StructField("tolls_amount", DoubleType()),
    StructField("total_amount", DoubleType()),])

# Read Kafka Stream and Features

This block reads taxi ride events from the Kafka stream, parses the JSON payload using the defined schema, and enriches each event with spatial (grid cell) and temporal (day of week, hour) features. These transformations prepare the real-time data for hotspot detection and comparison with historical baselines.

In [9]:
kafka_raw = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "taxis_json")
    .option("startingOffsets", "earliest")
    .load())

events = (kafka_raw
    .selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), taxi_schema).alias("t"))
    .select("t.*")
    .withColumn("pickup_ts", to_timestamp("pickup_datetime"))
    .dropna(subset=["pickup_ts","pickup_latitude","pickup_longitude"])
    .withColumn("pickup_grid", grid_udf(col("pickup_latitude"), col("pickup_longitude")))
    .withColumn("cell_id", concat_ws("_", col("pickup_grid.row"), col("pickup_grid.col")))
    .filter((col("pickup_grid.row") > 0) & (col("pickup_grid.row") < 300) &
            (col("pickup_grid.col") > 0) & (col("pickup_grid.col") < 300))
    .withColumn("dow", dayofweek("pickup_ts"))
    .withColumn("hour", hour("pickup_ts")))

# Parse Kafka JSON

This block parses and enriches the Kafka stream with spatial (grid cells) and temporal (day of week, hour) features, preparing each ride event for real-time aggregation and hotspot detection. By structuring the stream data, it allows comparison with historical baselines.

In [32]:
from pyspark.sql.functions import from_json, to_timestamp, concat_ws, dayofweek, hour, col

pickup_events = (kafka_raw
    .selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), taxi_schema).alias("t"))
    .select("t.*")
    .withColumn("pickup_ts", to_timestamp("pickup_datetime"))
    .dropna(subset=["pickup_ts","pickup_latitude","pickup_longitude"])
    .withColumn("pickup_grid", grid_udf(col("pickup_latitude"), col("pickup_longitude")))
    .withColumn("pick_cell_id", concat_ws("_", col("pickup_grid.row"), col("pickup_grid.col")))
    .filter((col("pickup_grid.row") > 0) & (col("pickup_grid.row") < 300) &
            (col("pickup_grid.col") > 0) & (col("pickup_grid.col") < 300))
    .withColumn("dow", dayofweek("pickup_ts"))
    .withColumn("hour", hour("pickup_ts")))

dropoff_events_complete = (kafka_raw
    .selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), taxi_schema).alias("t"))
    .select("t.*")
    .withColumn("dropoff_ts", to_timestamp("dropoff_datetime"))
    .dropna(subset=["dropoff_ts","dropoff_latitude","dropoff_longitude"])
    .withColumn("dropoff_grid", grid_udf(col("dropoff_latitude"), col("dropoff_longitude")))
    .withColumn("drop_cell_id", concat_ws("_", col("dropoff_grid.row"), col("dropoff_grid.col")))
    .withColumn("dow", dayofweek("dropoff_ts"))
    .withColumn("hour", hour("dropoff_ts")))

# Geographic Analysis

## Most Frequent Areas

In [14]:
#Pick up hotspots
pickups_hotspots = (pickup_events
    .withWatermark("pickup_ts", "30 minutes")
    .groupBy(window(col("pickup_ts"), "10 minutes", "1 minute").alias("w"), col("pick_cell_id"))
    .agg(count("*").alias("rides_10m"))
)

from pyspark.sql.functions import col, desc

def show_hotspots(df, epoch):
    try:
        out = (
            df.select(
                col("w.start").alias("start"),
                col("w.end").alias("end"),
                "pick_cell_id",
                "rides_10m"
            )
            .orderBy(desc("rides_10m"))
            .limit(20)
        )

        print(f"\n=== PICK UP HOTSPOTS epoch {epoch} ===")
        out.show(truncate=False)

    except Exception as e:
        print(f"[show_hotspots] epoch {epoch} interrupted/error: {e}")

import uuid
CHK = f"/tmp/chk_hotspots_{uuid.uuid4().hex}"

q = (pickups_hotspots.writeStream
    .outputMode("update")
    .foreachBatch(show_hotspots)
    .option("checkpointLocation", CHK)
    .trigger(processingTime="10 seconds")
    .start()
)
q.awaitTermination(40)
q.stop()


=== HOTSPOTS epoch 0 ===
+-------------------+-------------------+------------+---------+
|start              |end                |pick_cell_id|rides_10m|
+-------------------+-------------------+------------+---------+
|2025-12-21 13:28:00|2025-12-21 13:38:00|161_154     |18       |
|2025-12-21 13:33:00|2025-12-21 13:43:00|164_155     |16       |
|2025-12-21 13:30:00|2025-12-21 13:40:00|161_154     |16       |
|2025-12-21 13:31:00|2025-12-21 13:41:00|161_154     |16       |
|2025-12-21 13:27:00|2025-12-21 13:37:00|161_154     |16       |
|2025-12-21 13:26:00|2025-12-21 13:36:00|161_154     |16       |
|2025-12-21 13:29:00|2025-12-21 13:39:00|161_154     |16       |
|2025-12-21 13:32:00|2025-12-21 13:42:00|164_155     |15       |
|2025-12-21 13:32:00|2025-12-21 13:42:00|163_155     |15       |
|2025-12-21 13:34:00|2025-12-21 13:44:00|164_155     |15       |
|2025-12-21 13:32:00|2025-12-21 13:42:00|159_158     |15       |
|2025-12-21 13:31:00|2025-12-21 13:41:00|163_155     |15       |

In [26]:
#Drop off hotspots
from pyspark.sql.functions import col, desc, count, window
import time

dropoff_hotspots = (dropoff_events
    .withWatermark("dropoff_ts", "30 minutes")
    .groupBy(window(col("dropoff_ts"), "10 minutes", "1 minute").alias("w"), col("drop_cell_id"))
    .agg(count("*").alias("rides_10m"))
)

def show_dropoff_hotspots(batch_df, batch_id):
    try:
        if batch_df.isEmpty():
            print(f"[Batch {batch_id}] Empty")
            return

        top_hotspots = batch_df.collect()

        if not top_hotspots:
            return

        sorted_hotspots = sorted(top_hotspots, key=lambda x: x['rides_10m'], reverse=True)[:20]

        print(f"\n{'='*80}")
        print(f"=== DROPOFF HOTSPOTS epoch {batch_id} ===")
        print(f"{'='*80}")
        print(f"{'Window Start':<20} {'Window End':<20} {'Cell ID':<12} {'Rides'}")
        print(f"{'-'*80}")

        for row in sorted_hotspots:
            start = row['w']['start'].strftime('%Y-%m-%d %H:%M')
            end = row['w']['end'].strftime('%Y-%m-%d %H:%M')
            cell = row['drop_cell_id']
            rides = row['rides_10m']
            print(f"{start:<20} {end:<20} {cell:<12} {rides:>6d}")

    except Exception as e:
        print(f"[Batch {batch_id}] Error: {str(e)[:100]}")

import os
import shutil
CHK = "/tmp/chk_dropoff_hotspots"
if os.path.exists(CHK):
    shutil.rmtree(CHK)

q = (dropoff_hotspots.writeStream
    .outputMode("update")
    .foreachBatch(show_dropoff_hotspots)
    .option("checkpointLocation", CHK)
    .trigger(processingTime="30 seconds")
    .start()
)

try:
    time.sleep(300)
except KeyboardInterrupt:
    print("\ Stopping stream")

q.stop()
print("Stream closed with sucess!")

<>:57: SyntaxWarning: invalid escape sequence '\ '
<>:57: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-4228358765.py:57: SyntaxWarning: invalid escape sequence '\ '
  print("\ Stopping stream")



=== DROPOFF HOTSPOTS epoch 0 ===
Window Start         Window End           Cell ID      Rides
--------------------------------------------------------------------------------
2025-12-21 14:36     2025-12-21 14:46     159_157          75
2025-12-21 14:35     2025-12-21 14:45     159_157          74
2025-12-21 13:51     2025-12-21 14:01     161_154          73
2025-12-21 14:37     2025-12-21 14:47     159_157          73
2025-12-21 13:50     2025-12-21 14:00     161_154          71
2025-12-21 13:52     2025-12-21 14:02     161_154          70
2025-12-21 14:34     2025-12-21 14:44     159_157          68
2025-12-21 13:54     2025-12-21 14:04     161_154          67
2025-12-21 13:53     2025-12-21 14:03     161_154          67
2025-12-21 14:33     2025-12-21 14:43     159_157          65
2025-12-21 14:12     2025-12-21 14:22     159_157          64
2025-12-21 14:59     2025-12-21 15:09     159_157          64
2025-12-21 15:00     2025-12-21 15:10     159_157          64
2025-12-21 13:55  

In this section, we analyze taxi pick up and dropoff activity in real time by identifying the most active cells across the city. Using Spark Structured Streaming, we process incoming events in 10-minute sliding windows with 1-minute increments, computing the number of dropoffs in each spatial cell during each window.

In general, the values obtained from pickups and dropoffs fluctuate from epoch to epoch. This is normal because each batch only shows the incremental updates within the 10-minute windows.

In Assignment #1, the most common cells for both pickups and dropoffs were: (161,154), (160,157), (161,156), (160,154), (159,157), (161,157), (158,156), and (159,155). Some of these cells frequently appear in the top 20 of the streaming batches, such as (161,154), (160,154), (159,157), among others. While these cells are recorded in the streaming process, their values are much smaller because they reflect recent activity only, not cumulative totals.

These results demonstrate a clear consistency between batch and streaming data. The areas that are historically the most active in the cumulative dataset are also the ones that show the highest activity in the real-time streaming batches. This indicates that the streaming data accurately captures the same spatial patterns observed in the historical dataset, while providing the added benefit of immediate visibility into current trends and fluctuations.

## Most Frequent Routes

In [25]:
routes = (events
    .dropna(subset=["dropoff_latitude","dropoff_longitude"])
    .withColumn("drop_grid", grid_udf(col("dropoff_latitude"), col("dropoff_longitude")))
    .withColumn("drop_cell_id", concat_ws("_", col("drop_grid.row"), col("drop_grid.col")))
    .filter((col("drop_grid.row") > 0) & (col("drop_grid.row") < 300) &
            (col("drop_grid.col") > 0) & (col("drop_grid.col") < 300))
    .withWatermark("pickup_ts", "30 minutes")
    .groupBy(window(col("pickup_ts"), "15 minutes", "1 minute").alias("w"),
             col("cell_id").alias("pickup_cell"),
             col("drop_cell_id").alias("dropoff_cell"))
    .agg(count("*").alias("rides_15m"))
)

def show_routes(df, epoch):
    out = (df.select(col("w.start").alias("start"), col("w.end").alias("end"),
                     "pickup_cell","dropoff_cell","rides_15m")
            .orderBy(col("rides_15m").desc())
            .limit(10))
    print(f"\n=== ROUTES epoch {epoch} ===")
    out.show(truncate=False)

!rm -rf /tmp/chk_routes
q = (routes.writeStream
    .outputMode("update")
    .foreachBatch(show_routes)
    .option("checkpointLocation", "/tmp/chk_routes")
    .trigger(processingTime="10 seconds")
    .start()
)
q.awaitTermination(40)
q.stop()


=== ROUTES epoch 0 ===
+-------------------+-------------------+-----------+------------+---------+
|start              |end                |pickup_cell|dropoff_cell|rides_15m|
+-------------------+-------------------+-----------+------------+---------+
|2025-12-21 13:53:00|2025-12-21 14:08:00|161_154    |160_157     |7        |
|2025-12-21 13:54:00|2025-12-21 14:09:00|161_154    |160_157     |7        |
|2025-12-21 14:41:00|2025-12-21 14:56:00|161_154    |160_157     |7        |
|2025-12-21 14:43:00|2025-12-21 14:58:00|161_154    |161_157     |6        |
|2025-12-21 14:08:00|2025-12-21 14:23:00|154_160    |156_159     |6        |
|2025-12-21 14:13:00|2025-12-21 14:28:00|160_155    |161_154     |6        |
|2025-12-21 13:52:00|2025-12-21 14:07:00|161_154    |160_157     |6        |
|2025-12-21 14:12:00|2025-12-21 14:27:00|162_156    |164_155     |6        |
|2025-12-21 15:46:00|2025-12-21 16:01:00|155_156    |154_156     |6        |
|2025-12-21 14:22:00|2025-12-21 14:37:00|160_154    

To better understand urban mobility patterns, we also analyze the most frequent routes in real time, defined as the pair of pickup and dropoff cells for each ride. The streaming query aggregates rides in 15-minute sliding windows with 1-minute increments, counting the number of rides for each pickup→dropoff pair.

Again, the most frequent routes recorded in each epoch show some variability among themselves, which is expected since each batch incrementally processes each window.

The results show that, in a streaming context, the most frequent routes have low absolute volumes, reflecting the use of short temporal windows. However, these routes reveal consistent patterns within those windows and, in several cases, persist across consecutive batches, indicating that they are not merely noise but rather short-term behaviors. Some of the routes that end up being recorded most frequently include, for example, (161,154)→(161,156), (156,174)→(171,150), (159,158)→(156,161), and (161,154)→(160,157).

Compared to the batch approach, it can be observed that many of the routes detected in real time correspond to the same urban regions identified as relevant in the historical data, although with finer spatial variations that become diluted when analyzing the full dataset.

This comparison highlights that batch processing provides a notion of what is “normal” or typical in the urban transport system, while streaming makes it possible to identify temporary deviations, micro-patterns, and local fluctuations that are only visible in real time. Thus, streaming knowledge enriches batch analysis by allowing current behavior to be contrasted with historical patterns, paving the way for anomaly analysis, event detection, or sudden changes in demand.

### **Enriching the Batch Analysis with Real-Time Streaming Data**

In Assignment #1, the analysis focused on identifying long-term geographical patterns in taxi activity, revealing consistent pickup and dropoff hotspots and frequent routes across the city. This batch analysis provides a notion of what is normal or typical in terms of urban mobility.

By incorporating real-time streaming data in Assignment #2, this analysis is enriched with a temporal dimension that allows observation of how these patterns evolve in short time windows. While the same geographical hotspots identified in the historical dataset also appear in the streaming analysis, their activity levels fluctuate across batches, reflecting current and localized demand rather than cumulative totals.

This demonstrates that real-time data refines the batch analysis by adding immediacy and sensitivity to short-term changes, enabling the detection of transient behaviors that are invisible when analyzing the full dataset at once.

### **Potential Applications Enabled by Real-Time Knowledge**

The availability of real-time streaming data enables applications that are not possible with batch processing alone. For example:

**Dynamic taxi repositioning:** taxis could be redirected toward areas that are experiencing unusually high pickup demand compared to their historical baseline.

**Event or anomaly detection:** sudden spikes in pickups or dropoffs in specific cells may indicate special events, traffic disruptions, or weather-related effects.

**Urban monitoring:** transportation authorities could monitor congestion or demand surges as they happen, rather than retrospectively.

In this context, the batch analysis acts as a historical reference, while the streaming analysis allows current behavior to be continuously compared against what is considered normal.

### **Comparison Between Historical and Streaming Patterns**

The comparison between batch and streaming results shows strong spatial consistency when evaluated through abstract statistical indicators such as frequency, ranking stability, and variability. Cells such as (161,154), (160,157), and (161,156) appear repeatedly as the most active regions in both datasets, indicating a stable spatial distribution of demand.

However, while batch processing reflects long-term averages and cumulative counts, the streaming data captures short-term fluctuations, resulting in significantly lower absolute counts but higher temporal variability across consecutive windows. This increased variance is an expected consequence of observing the system over shorter time intervals.

Similarly, frequent routes observed in the streaming context correspond to the same urban regions identified in the historical dataset, preserving the overall spatial structure while revealing finer-grained temporal dynamics. From a statistical perspective, this shows that real-time streams maintain the same underlying distribution as the archived data, but with increased dispersion and volatility due to the reduced aggregation period.


# Temporal Analysis

## Number of rides per hour of the day

In [34]:
#Pick ups
import uuid
from pyspark.sql.functions import col, window, count

spark.conf.set("spark.sql.shuffle.partitions", "1")

for s in list(spark.streams.active):
    s.stop()
for s in list(spark.streams.active):
    try: s.awaitTermination(10)
    except: pass

rides_by_hour = (events
    .withWatermark("pickup_ts", "30 minutes")
    .groupBy(window(col("pickup_ts"), "60 minutes", "5 minutes").alias("w"), col("hour"))
    .agg(count("*").alias("rides_60m"))
)

def show_hour(df, epoch):
    out = (df.select(col("w.end").alias("time"), "hour", "rides_60m")
             .orderBy(col("time").desc(), col("rides_60m").desc())
             .limit(30))
    print(f"\n=== PICK UP RIDES BY HOUR epoch {epoch} ===")
    out.show(truncate=False)

chk = f"/tmp/chk_hour_{uuid.uuid4().hex}"
print("checkpointLocation =", chk)

q = (rides_by_hour.writeStream
    .outputMode("update")
    .foreachBatch(show_hour)
    .option("checkpointLocation", chk)
    .trigger(processingTime="10 seconds")
    .start()
)

q.awaitTermination(25)
q.stop()

checkpointLocation = /tmp/chk_hour_1c7495dcd94f4044b49da1342960d19f

=== PICK UP RIDES BY HOUR epoch 0 ===
+-------------------+----+---------+
|time               |hour|rides_60m|
+-------------------+----+---------+
|2025-12-21 17:55:00|16  |13       |
|2025-12-21 17:50:00|16  |111      |
|2025-12-21 17:45:00|16  |494      |
|2025-12-21 17:40:00|16  |1279     |
|2025-12-21 17:35:00|16  |2205     |
|2025-12-21 17:30:00|16  |3163     |
|2025-12-21 17:25:00|16  |3881     |
|2025-12-21 17:20:00|16  |4626     |
|2025-12-21 17:15:00|16  |5566     |
|2025-12-21 17:10:00|16  |6492     |
|2025-12-21 17:05:00|16  |7353     |
|2025-12-21 17:00:00|16  |7991     |
|2025-12-21 16:55:00|16  |7978     |
|2025-12-21 16:55:00|15  |682      |
|2025-12-21 16:50:00|16  |7880     |
|2025-12-21 16:50:00|15  |1489     |
|2025-12-21 16:45:00|16  |7497     |
|2025-12-21 16:45:00|15  |2446     |
|2025-12-21 16:40:00|16  |6712     |
|2025-12-21 16:40:00|15  |3117     |
+-------------------+----+---------+
only 

In [33]:
#Dropoff
import uuid
from pyspark.sql.functions import col, window, count, hour, dayofweek

spark.conf.set("spark.sql.shuffle.partitions", "1")
for s in list(spark.streams.active):
    s.stop()
for s in list(spark.streams.active):
    try:
        s.awaitTermination(10)
    except:
        pass

rides_by_hour = (dropoff_events_complete
    .withWatermark("dropoff_ts", "30 minutes")
    .groupBy(window(col("dropoff_ts"), "60 minutes", "5 minutes").alias("w"), col("hour"))
    .agg(count("*").alias("rides_60m")))

def show_hour(df, epoch):
    try:
        if df.isEmpty():
            print(f"[Epoch {epoch}] Vazio - aguardando dados...")
            return

        out = (df.select(col("w.end").alias("time"), "hour", "rides_60m")
                 .orderBy(col("time").desc(), col("rides_60m").desc())
                 .limit(30))
        print(f"\n=== DROP OFF RIDES BY HOUR epoch {epoch} ===")
        out.show(truncate=False)
    except Exception as e:
        print(f"[Epoch {epoch}] Erro: {str(e)[:100]}")

chk = f"/tmp/chk_dropoff_hour_{uuid.uuid4().hex}"
print("checkpointLocation =", chk)

q = (rides_by_hour.writeStream
    .outputMode("update")
    .foreachBatch(show_hour)
    .option("checkpointLocation", chk)
    .trigger(processingTime="10 seconds")
    .start()
)

q.awaitTermination(25)
q.stop()

checkpointLocation = /tmp/chk_dropoff_hour_aa6b3b0e7d3744ef85d1a31f174cf60d

=== DROP OFF RIDES BY HOUR epoch 0 ===
+-------------------+----+---------+
|time               |hour|rides_60m|
+-------------------+----+---------+
|2025-12-21 17:55:00|16  |81       |
|2025-12-21 17:50:00|16  |913      |
|2025-12-21 17:45:00|16  |2264     |
|2025-12-21 17:40:00|16  |3333     |
|2025-12-21 17:35:00|16  |4381     |
|2025-12-21 17:30:00|16  |4564     |
|2025-12-21 17:25:00|16  |5490     |
|2025-12-21 17:20:00|16  |6698     |
|2025-12-21 17:15:00|16  |7698     |
|2025-12-21 17:10:00|16  |8491     |
|2025-12-21 17:05:00|16  |8738     |
|2025-12-21 17:00:00|16  |9556     |
|2025-12-21 16:55:00|16  |9475     |
|2025-12-21 16:55:00|15  |1077     |
|2025-12-21 16:50:00|16  |8643     |
|2025-12-21 16:50:00|15  |1947     |
|2025-12-21 16:45:00|16  |7292     |
|2025-12-21 16:45:00|15  |2292     |
|2025-12-21 16:40:00|16  |6223     |
|2025-12-21 16:40:00|15  |3617     |
+-------------------+----+-------

In Assigment #1, we performed a batch processing analysis to study temporal patterns of taxi pickups and dropoffs. Specifically, we computed the total number of rides per hour, obtaining a clear pattern of daily demand. In the streaming analysis, we processed the same dataset, but in a continuous manner, using 60-minute sliding windows to aggregate ride counts at each epoch. Each epoch represents a micro-batch of incoming events, allowing us to track the number of pickups and dropoffs over the past hour, updated every few minutes. This method enables us to observe demand evolution dynamically, something that batch processing cannot provide.

During the late afternoon hours, our streaming results show that pickup counts gradually increase from around 4,600 rides in one 60-minute window at 17:20, up to approximately 8,000 rides at 17:00. Dropoffs display similar dynamics, with slightly higher counts due to the natural delay between pickup and dropoff times, reflecting ongoing trips that started in previous windows. This temporal desynchronization between pickups and dropoffs highlights a phenomenon that batch analysis alone cannot capture: the propagation of trips through time, which is crucial for understanding real-time supply-demand balance and ride duration distributions.

### **Enriching the Batch Analysis with Real-Time Streaming Data**
Using the streaming results, we can not only confirm the historical patterns detected in batch processing, but also detect deviations from typical behavior as they happen. For instance, while batch analysis identifies peak demand hours between 18:00 and 20:00, streaming allows us to see exactly when demand starts to rise, how fast it grows, and whether current conditions match the expected trend.

This is particularly important for operational planning, as it enables proactive decision-making, such as allocating more drivers to high-demand areas before the peak occurs, or identifying early drops in demand that might indicate unusual conditions, like weather events or traffic incidents.

### **Potential Applications Enabled by Real-Time Knowledge**
There are several examples where streaming insights provide clear advantages: dynamic taxi dispatching, surge pricing, and immediate detection of abnormal events. Real-time information allows taxi companies or ride-hailing platforms to adjust driver availability on the fly, redistribute vehicles geographically, and improve overall service responsiveness.

Moreover, by monitoring pickup and dropoff patterns as they evolve, platforms can identify emerging congestion points, adjust routing algorithms, and optimize operational efficiency. In short, streaming data enables actionable insights that cannot be derived from batch data alone, turning historical knowledge into real-time operational intelligence.

### **Comparison Between Historical and Streaming Patterns**
By examining sliding windows of 60 minutes, we see that streaming counts approximately match the expected patterns from batch analysis, though scaled down to shorter intervals. For example, during the 16:00–17:00 window, the number of pickups in streaming ranges around 8,000, while the batch total for the same hour is approximately 68,000.

This discrepancy is not a flaw, but a natural consequence of window size, aggregation frequency, and the continuous arrival of events. More importantly, the patterns observed—rising demand in late afternoon and high dropoff counts slightly lagging pickups—are consistent with historical trends, validating the real-time approach. Streaming provides fine-grained temporal resolution, revealing micro-patterns within hours that batch analysis smooths over.

When discussing discrepancies between batch and streaming, several factors should be highlighted. First, batch aggregates data over 24 hours, giving a stable, cumulative picture of demand, while streaming aggregates data in short, sliding windows, making the results more sensitive to temporal fluctuations. Second, the desynchronization between pickups and dropoffs in streaming windows is more evident, reflecting actual travel durations, whereas batch counts obscure these dynamics. Third, the speedup factor applied to simulate faster-than-real-time streaming introduces compressed time intervals, but trip durations remain accurate, producing waves of activity that reflect realistic traffic and ride patterns. Finally, minor differences between consecutive epochs demonstrate smooth convergence of counts as new data flows in, a phenomenon that batch processing cannot illustrate.





# AI Addendum

Artificial Intelligence (AI) was an important and essential tool in the development of this work, particularly in the coding part. Initially, the document containing the project guidelines, "SPBD2526_Proj2.ipynb," was submitted to the AI tool ChatGPT. Information regarding Assignment #1 was also provided, specifically that in this first assignment, geographic patterns were studied through the most common pickup and dropoff locations, as well as the most frequent routes. Temporal patterns were also analyzed, including the hours with the highest number of trips throughout the day, the days with the most trips during the week, and the months with the highest number of trips during the year.

For Assignment #2, the goal was to conduct a similar analysis using streaming and incremental processing. This involved identifying pickup and dropoff hotspots within defined batch windows, analyzing the most frequent routes, and incorporating the number of trips per hour to complement the spatial pattern analysis conducted in the previous assignment.

During this process, it was observed that ChatGPT frequently provided code solutions focused primarily on pickup locations—both in identifying the most common locations and the busiest hours of the day. Generating accurate code for dropoff locations, however, proved more challenging, as multiple attempts were required to resolve errors. When these errors occurred, the code along with the corresponding error messages was submitted back to ChatGPT for review and correction. After careful testing and refinement, the most correct approach was selected to ensure the analysis was accurate and complete.

In conclusion, AI served as a valuable tool for accelerating coding and problem-solving, but careful validation and iterative refinement were necessary to achieve reliable results, particularly for the more complex aspects of the analysis, such as dropoff patterns. This approach demonstrates how AI can support analytical work while still requiring human oversight to ensure correctness and completeness.